In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Load processed data

In [2]:

df = pd.read_csv("../data/processed_features.csv")

print("Shape:", df.shape)

display(df.head())

Shape: (3600, 18)


,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms,time_since_last_sms_hours,time_since_last_sms_days,recent_bluetooth_count_48h,hour_of_day,day_of_week,weekend
0,184,300,301,1,0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,0.051111,Sunday,1
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,NaN,7,2.0,0.000556,0.000023,0,1.088889,Sunday,1
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,NaN,1,15.0,0.004167,0.000174,0,1.561944,Sunday,1
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0,2.570000,Sunday,1
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,NaN,28,130.0,0.036111,0.001505,0,4.296111,Sunday,1


In [3]:
# Select features for clustering

cluster_features = [
    "previous_calls_between",
    "previous_answer_rate_between",
    "time_since_last_call_hours",
    "receiver_previous_answer_rate",
    "recent_sms_count_48h",
    "time_since_last_sms_hours",
    "recent_bluetooth_count_48h"
]

X = df[cluster_features].copy()

display(X.head())

,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms_hours,recent_bluetooth_count_48h
0,0,NaN,NaN,NaN,0,NaN,0
1,0,NaN,NaN,NaN,7,0.000556,0
2,1,1.0,1.510833,NaN,1,0.004167,0
3,0,NaN,NaN,NaN,0,NaN,0
4,0,NaN,NaN,NaN,28,0.036111,0


In [4]:
# Handle missing history

X["no_previous_call_history"] = (
    df["previous_calls_between"] == 0
).astype(int)

X["no_previous_sms_history"] = (
    df["time_since_last_sms_hours"].isna()
).astype(int)

X["no_receiver_history"] = (
    df["receiver_previous_answer_rate"].isna()
).astype(int)

In [5]:
# Missing-history indicators have already been created.
# Missing numerical values are filled with the median so that
# "no history" is not confused with a genuine value of zero.

X["previous_answer_rate_between"] = (
    X["previous_answer_rate_between"]
    .fillna(X["previous_answer_rate_between"].median())
)

X["time_since_last_call_hours"] = (
    X["time_since_last_call_hours"]
    .fillna(X["time_since_last_call_hours"].median())
)

X["receiver_previous_answer_rate"] = (
    X["receiver_previous_answer_rate"]
    .fillna(X["receiver_previous_answer_rate"].median())
)

X["time_since_last_sms_hours"] = (
    X["time_since_last_sms_hours"]
    .fillna(X["time_since_last_sms_hours"].median())
)

print("Missing values remaining:")
print(X.isna().sum())

Missing values remaining:
previous_calls_between           0
previous_answer_rate_between     0
time_since_last_call_hours       0
receiver_previous_answer_rate    0
recent_sms_count_48h             0
time_since_last_sms_hours        0
recent_bluetooth_count_48h       0
no_previous_call_history         0
no_previous_sms_history          0
no_receiver_history              0
dtype: int64


In [6]:
X_transformed = X.copy()

skewed_features = [
    "previous_calls_between",
    "time_since_last_call_hours",
    "recent_sms_count_48h",
    "time_since_last_sms_hours",
    "recent_bluetooth_count_48h"
]

for feature in skewed_features:
    X_transformed[feature] = np.log1p(X_transformed[feature])

display(X_transformed.describe().T)

,count,mean,std,min,25%,50%,75%,max
previous_calls_between,3600.0,1.744909,1.229654,0.000000,0.693147,1.609438,2.639057,4.867534
previous_answer_rate_between,3600.0,0.905704,0.201894,0.000000,0.894737,1.000000,1.000000,1.000000
time_since_last_call_hours,3600.0,1.558531,1.787689,0.000555,0.083754,0.683938,3.048967,6.444251
receiver_previous_answer_rate,3600.0,0.913989,0.189752,0.000000,0.928571,1.000000,1.000000,1.000000
recent_sms_count_48h,3600.0,1.286953,1.350463,0.000000,0.000000,1.098612,2.302585,5.789960
time_since_last_sms_hours,3600.0,2.168321,1.736555,0.000278,0.519579,1.862227,3.389418,6.462295
recent_bluetooth_count_48h,3600.0,2.295105,2.235847,0.000000,0.000000,2.564949,4.418841,6.347389
no_previous_call_history,3600.0,0.172500,0.377867,0.000000,0.000000,0.000000,0.000000,1.000000
no_previous_sms_history,3600.0,0.206389,0.404769,0.000000,0.000000,0.000000,0.000000,1.000000
no_receiver_history,3600.0,0.133333,0.339982,0.000000,0.000000,0.000000,0.000000,1.000000


In [7]:
print("Shape used for clustering:", X.shape)

display(X.describe().T)

Shape used for clustering: (3600, 10)


,count,mean,std,min,25%,50%,75%,max
previous_calls_between,3600.0,11.190556,17.788774,0.000000,1.000000,4.000000,13.000000,129.000000
previous_answer_rate_between,3600.0,0.905704,0.201894,0.000000,0.894737,1.000000,1.000000,1.000000
time_since_last_call_hours,3600.0,28.986615,71.109584,0.000556,0.087361,0.981667,20.093542,628.075556
receiver_previous_answer_rate,3600.0,0.913989,0.189752,0.000000,0.928571,1.000000,1.000000,1.000000
recent_sms_count_48h,3600.0,9.184444,21.139224,0.000000,0.000000,2.000000,9.000000,326.000000
time_since_last_sms_hours,3600.0,39.069597,80.952639,0.000278,0.681319,5.438056,28.648750,639.529444
recent_bluetooth_count_48h,3600.0,60.740000,99.540884,0.000000,0.000000,12.000000,82.000000,570.000000
no_previous_call_history,3600.0,0.172500,0.377867,0.000000,0.000000,0.000000,0.000000,1.000000
no_previous_sms_history,3600.0,0.206389,0.404769,0.000000,0.000000,0.000000,0.000000,1.000000
no_receiver_history,3600.0,0.133333,0.339982,0.000000,0.000000,0.000000,0.000000,1.000000


In [8]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_transformed)

print("Scaled data shape:", X_scaled.shape)

Scaled data shape: (3600, 10)


In [9]:
results = []

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    silhouette = silhouette_score(
        X_scaled,
        labels
    )

    results.append({
        "k": k,
        "inertia": kmeans.inertia_,
        "silhouette_score": silhouette
    })

cluster_evaluation = pd.DataFrame(results)

display(cluster_evaluation)

,k,inertia,silhouette_score
0,2,28269.082338,0.265311
1,3,23976.382574,0.225959
2,4,20548.452681,0.232726
3,5,18388.818702,0.248649
4,6,16794.883746,0.228457
5,7,15453.488172,0.246757
6,8,14513.963263,0.240036


Final K-means model:

In [10]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans.fit_predict(X_scaled)

df["cluster"] = cluster_labels

print(df["cluster"].value_counts().sort_index())

cluster
0     984
1    2616
Name: count, dtype: int64


In [11]:
cluster_profiles = (
    df.groupby("cluster")[cluster_features]
    .mean()
    .round(3)
)

display(cluster_profiles)

,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms_hours,recent_bluetooth_count_48h
cluster,,,,,,,
0,0.745,0.928,27.893,0.921,1.733,49.240,24.619
1,15.120,0.880,35.789,0.896,11.987,47.633,74.327


In [12]:
cluster_outcomes = (
    df.groupby("cluster")["answered"]
    .agg(
        number_of_calls="count",
        answered_rate="mean"
    )
)

cluster_outcomes["missed_rate"] = (
    1 - cluster_outcomes["answered_rate"]
)

display(cluster_outcomes.round(3))

,number_of_calls,answered_rate,missed_rate
cluster,,,
0,984,0.892,0.108
1,2616,0.901,0.099


Median cluster profiles

In [13]:

cluster_profiles_median = (
    df.groupby("cluster")[cluster_features]
    .median()
    .round(3)
)

display(cluster_profiles_median)

,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms_hours,recent_bluetooth_count_48h
cluster,,,,,,,
0,0.0,1.0,0.196,1.0,0.0,3.525,0.0
1,8.0,1.0,1.319,1.0,4.0,5.961,24.0


call outcome by cluster

In [16]:
cluster_outcomes = (
    df.groupby("cluster")["answered"]
    .agg(
        number_of_calls="count",
        answered_rate="mean"
    )
)

cluster_outcomes["missed_rate"] = (
    1 - cluster_outcomes["answered_rate"]
)

display(cluster_outcomes.round(3))

,number_of_calls,answered_rate,missed_rate
cluster,,,
0,984,0.892,0.108
1,2616,0.901,0.099


In [17]:
cluster_outcome_counts = pd.crosstab(
    df["cluster"],
    df["answered"]
)

cluster_outcome_counts.columns = [
    "missed",
    "answered"
]

display(cluster_outcome_counts)

,missed,answered
cluster,,
0,106,878
1,260,2356


Secondary clustering with k=5

In [18]:
kmeans_5 = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=20
)

cluster_labels_5 = kmeans_5.fit_predict(X_scaled)

df["cluster_5"] = cluster_labels_5

print("Cluster sizes:")
print(df["cluster_5"].value_counts().sort_index())

Cluster sizes:
cluster_5
0     847
1     563
2     441
3    1503
4     246
Name: count, dtype: int64


In [20]:
cluster_profiles_5 = (
    df.groupby("cluster_5")[cluster_features]
    .median()
    .round(3)
)

display(cluster_profiles_5)

,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms_hours,recent_bluetooth_count_48h
cluster_5,,,,,,,
0,6.0,1.000,22.090,1.0,0.0,92.054,1.0
1,1.0,1.000,0.255,1.0,0.0,5.265,1.0
2,0.0,1.000,0.500,NaN,0.0,2.088,0.0
3,11.0,1.000,0.519,1.0,10.0,0.864,39.0
4,3.0,0.367,0.124,0.5,1.0,2.534,0.0


In [21]:
cluster_outcomes_5 = (
    df.groupby("cluster_5")["answered"]
    .agg(
        number_of_calls="count",
        answered_rate="mean"
    )
)

cluster_outcomes_5["missed_rate"] = (
    1 - cluster_outcomes_5["answered_rate"]
)

display(cluster_outcomes_5.round(3))

,number_of_calls,answered_rate,missed_rate
cluster_5,,,
0,847,0.919,0.081
1,563,0.893,0.107
2,441,0.887,0.113
3,1503,0.925,0.075
4,246,0.695,0.305


Missing-history profile by cluster

In [22]:
history_indicators = [
    "no_previous_call_history",
    "no_previous_sms_history",
    "no_receiver_history"
]

indicator_profiles_5 = (
    pd.DataFrame(
        X[history_indicators]
    )
    .assign(cluster_5=df["cluster_5"].values)
    .groupby("cluster_5")[history_indicators]
    .mean()
    .round(3)
)

display(indicator_profiles_5)

,no_previous_call_history,no_previous_sms_history,no_receiver_history
cluster_5,,,
0,0.000,0.000,0.015
1,0.485,0.796,0.000
2,0.769,0.501,1.000
3,0.003,0.000,0.007
4,0.016,0.301,0.065


k=5 summary

In [23]:
cluster_summary_5 = cluster_profiles_5.copy()

cluster_summary_5["number_of_calls"] = (
    df["cluster_5"].value_counts().sort_index()
)

cluster_summary_5["answered_rate"] = (
    df.groupby("cluster_5")["answered"].mean()
)

cluster_summary_5["missed_rate"] = (
    1 - cluster_summary_5["answered_rate"]
)

display(cluster_summary_5.round(3))

,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms_hours,recent_bluetooth_count_48h,number_of_calls,answered_rate,missed_rate
cluster_5,,,,,,,,,,
0,6.0,1.000,22.090,1.0,0.0,92.054,1.0,847,0.919,0.081
1,1.0,1.000,0.255,1.0,0.0,5.265,1.0,563,0.893,0.107
2,0.0,1.000,0.500,NaN,0.0,2.088,0.0,441,0.887,0.113
3,11.0,1.000,0.519,1.0,10.0,0.864,39.0,1503,0.925,0.075
4,3.0,0.367,0.124,0.5,1.0,2.534,0.0,246,0.695,0.305


Test association between cluster and call data

In [24]:
from scipy.stats import chi2_contingency

outcome_table_5 = pd.crosstab(
    df["cluster_5"],
    df["answered"]
)

outcome_table_5.columns = [
    "missed",
    "answered"
]

chi2, p_value, dof, expected = chi2_contingency(
    outcome_table_5
)

print("Chi-square:", round(chi2, 3))
print("Degrees of freedom:", dof)
print("p-value:", p_value)

display(outcome_table_5)

Chi-square: 127.954
Degrees of freedom: 4
p-value: 1.0664451488922507e-26


,missed,answered
cluster_5,,
0,69,778
1,60,503
2,50,391
3,112,1391
4,75,171


In [25]:
# Effect size: Cramer's V

n = outcome_table_5.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 /
    (
        n *
        min(
            outcome_table_5.shape[0] - 1,
            outcome_table_5.shape[1] - 1
        )
    )
)

print("Cramer's V:", round(cramers_v, 3))

Cramer's V: 0.189


### Interpretation of the clustering results

The two-cluster solution achieved the highest silhouette score (0.265) and mainly separated calls with lower and higher levels of previous interaction. However, the difference in call outcome between these two broad profiles was small.

To investigate more detailed interaction profiles, a five-cluster solution was also examined. Although its silhouette score was slightly lower (0.249), it revealed more distinct interaction patterns.

In particular, Cluster 4 was characterized by lower previous answer rates, relatively limited SMS communication and little recent Bluetooth proximity. This cluster had a missed-call rate of 30.5%, compared with 7.5–11.3% in the other clusters.

A chi-square test indicated that call outcome was associated with cluster membership, χ²(4) = 127.954, p < 0.001. Cramér's V was 0.189, indicating that the association was present but not particularly strong.

These results suggest that interaction profiles are associated with call responsiveness. In particular, previous answering behaviour appears to distinguish the cluster with the highest proportion of missed calls.

In [26]:
clustering_results = df[
    [
        "timestamp",
        "caller",
        "callee",
        "answered",
        "cluster",
        "cluster_5"
    ]
].copy()

clustering_results.to_csv(
    "../data/clustering_results.csv",
    index=False
)

cluster_summary_5.to_csv(
    "../data/clustering_summary_5.csv"
)

print("Clustering results saved.")

Clustering results saved.
